# 🛰️ Field Shift: Step 4 - IoT Simulation & Zero-Hallucination RAG
**NASA Space Apps Challenge 2026**

This final notebook demonstrates the complete production pipeline:
1. **Rule Engine:** Building a local Crop Knowledge Base (JSON).
2. **Vector DB (FAISS):** Indexing the real FAO-56 Irrigation PDF for factual retrieval.
3. **IoT Simulation:** Simulating live sensor data and comparing it against the CatBoost AI Baseline.
4. **Resilient RAG:** A robust LLM pipeline using the new `google-genai` SDK with auto-fallback to handle server load, explaining the decision strictly based on FAO documents.

In [ ]:
# --- 1. Auto-install Required Packages ---
import subprocess
import sys

print("📦 Installing required packages for RAG (pypdf, langchain, faiss, google-genai)...")
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "-U",
    "pypdf", "langchain-community", "langchain-text-splitters",
    "langchain-huggingface", "faiss-cpu", "sentence-transformers", "google-genai"
])

import warnings
warnings.filterwarnings("ignore")
print("✅ Packages installed successfully.")

In [ ]:
# --- 2. Building Local Crop Knowledge Base ---
import json
import os
from google.colab import drive

print("⚙️ Building Local Crop Knowledge Base...")

# Ensure Drive is mounted
drive.mount('/content/drive')
project_path = '/content/drive/MyDrive/Field_Shift_NASA_Project'
os.makedirs(project_path, exist_ok=True)
json_path = os.path.join(project_path, 'crops_knowledge_base.json')

# Crop properties database
crops_db = {
    "Wheat": {"name_ar": "قمح", "water_need": "Medium", "nitrogen_fix": False, "root_depth": "Medium", "benefit": "Staple cash crop, good winter cover."},
    "Corn": {"name_ar": "ذرة", "water_need": "High", "nitrogen_fix": False, "root_depth": "Shallow", "benefit": "High yield and market value, but depletes soil nutrients."},
    "Alfalfa": {"name_ar": "برسيم حجازي", "water_need": "High", "nitrogen_fix": True, "root_depth": "Deep", "benefit": "Improves soil organic matter, fixes nitrogen, deep roots prevent erosion."},
    "Sorghum": {"name_ar": "ذرة رفيعة", "water_need": "Low", "nitrogen_fix": False, "root_depth": "Deep", "benefit": "Highly drought and heat resistant, excellent for water conservation."},
    "Lentils": {"name_ar": "عدس", "water_need": "Low", "nitrogen_fix": True, "root_depth": "Shallow", "benefit": "Drought tolerant, fixes soil nitrogen, good for dry seasons."}
}

# Save JSON
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(crops_db, f, ensure_ascii=False, indent=4)

print(f"✅ Crop Knowledge Base saved successfully at: {json_path}")

In [ ]:
# --- 3. Building FAISS Vector DB from Real FAO PDFs ---
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

def build_real_pdf_index():
    docs_folder = os.path.join(project_path, "real_docs")
    if not os.path.exists(docs_folder):
        os.makedirs(docs_folder)
        print(f"⚠️ Created '{docs_folder}' folder. Please put your FAO-56 PDF inside it and run again.")
        return None

    print(f"🚀 [1/4] Loading Real PDFs from '{docs_folder}'...")
    loader = PyPDFDirectoryLoader(docs_folder)
    docs = loader.load()

    if not docs:
        print("❌ No PDFs found! Please add the FAO PDF file to the folder.")
        return None

    print(f"📄 Loaded {len(docs)} pages from the FAO document.")

    print("✂️ [2/4] Splitting documents into semantic chunks...")
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
    chunks = text_splitter.split_documents(docs)
    print(f"✅ Created {len(chunks)} knowledge chunks from the real document.")

    print("🧠 [3/4] Generating Embeddings (all-MiniLM-L6-v2)...")
    embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

    print("🏗️ [4/4] Building and Saving FAISS Index...")
    vector_store = FAISS.from_documents(chunks, embeddings)
    
    output_dir = os.path.join(project_path, "faiss_index_real")
    vector_store.save_local(output_dir)
    print(f"✅ SUCCESS: Real Vector DB saved to '{output_dir}'!")
    return output_dir

faiss_path = build_real_pdf_index()

In [ ]:
# --- 4. IoT Simulation & Decision Engine ---
import pandas as pd
import catboost as cb

print("📡 Initiating IoT Early Warning Demo...")

# Load Model and Data
cb_model = cb.CatBoostRegressor()
cb_model.load_model(os.path.join(project_path, 'Field_Shift_CatBoost_Final.cbm'))
df = pd.read_csv(os.path.join(project_path, 'Field_Shift_Enhanced_100k.csv'))

features = [
    'Temperature_Avg', 'Relative_Humidity', 'Precipitation', 'NDVI',
    'LST_Celsius', 'Soil_Sand_pct', 'Soil_Clay_pct', 'Crop_Kc',
    'Thermal_Stress_Diff', 'VPD_kPa', 'Soil_Sponge_Index',
    'Month', 'DayOfYear', 'Rain_3d_sum', 'LST_5d_avg'
]

# Simulate Live Reading
sample_farm = df.sample(1, random_state=2026)
live_features = sample_farm[features]
farm_name = sample_farm['Location_Name'].values[0]

# Model predicts "Ideal" moisture
expected_moisture = cb_model.predict(live_features)[0]

# Simulate Severe Drought (Sensor reads 12% less than baseline)
live_sensor_reading = expected_moisture - 0.12

print("-" * 55)
print(f"📍 Location: {farm_name}")
print(f"🤖 AI Baseline (Optimal): {expected_moisture:.3f}")
print(f"📡 IoT Sensor (Actual): {live_sensor_reading:.3f}")
print("-" * 55)

# Decision Logic
threshold = 0.10
difference = expected_moisture - live_sensor_reading
crop_recommendation = ""

if difference > threshold:
    print("🚨 SEVERE WATER STRESS DETECTED!")
    crop_recommendation = "Alfalfa" # Selected based on JSON logic to heal soil
elif difference < -threshold:
    print("⚠️ OVER-IRRIGATION DETECTED!")
    crop_recommendation = "Corn"
else:
    print("✅ Farm Status: OPTIMAL.")
    crop_recommendation = "Wheat"

In [ ]:
# --- 5. Resilient RAG Pipeline (Google GenAI + FAISS) ---
import time
from google import genai

print("🚀 Running Field Shift Resilient RAG Pipeline...\n")

# 1. Initialize Client
my_key = "AQ.Ab8RN6J6_X0_CimVL9O3xRCD06PhMjGtJhY0kCa0PffO5uCqig" # Use your valid API key here
client = genai.Client(api_key=my_key)

# 2. Load FAISS Index
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vector_store = FAISS.load_local(faiss_path, embeddings, allow_dangerous_deserialization=True)
retriever = vector_store.as_retriever(search_kwargs={"k": 4})

# 3. Formulate Context Query based on IoT outputs
live_sponge_index = live_features['Soil_Sponge_Index'].values[0]
live_vpd = live_features['VPD_kPa'].values[0]

query = f"My AI model shows a Soil Sponge Index of {live_sponge_index:.2f} and a VPD of {live_vpd:.2f} kPa. The field is experiencing water stress. Should I plant Corn or {crop_recommendation}? Why?"
relevant_docs = retriever.invoke(query)
context_text = "\n".join([doc.page_content for doc in relevant_docs])

prompt = f"""
You are an expert agricultural AI assistant for the Field Shift project.
Use ONLY the following retrieved context from official NASA/FAO reports to answer the farmer's question.

Context:
{context_text}

Farmer's Scenario & Question:
{query}

Answer strictly based on the context:
"""

# 4. Smart Model Fallback System (503 Retry)
candidate_models = ["gemini-3.1-flash-lite", "gemini-2.5-flash", "gemini-2.0-flash"]

try:
    for m in client.models.list():
        name = m.name.replace("models/", "")
        if "gemini" in name and name not in candidate_models:
            candidate_models.append(name)
except Exception:
    pass

success = False
for model_name in candidate_models:
    if success:
        break
    for attempt in range(1, 4):  # 3 attempts per model
        try:
            print(f"📡 Calling {model_name} (Attempt {attempt}/3)...")
            response = client.models.generate_content(
                model=model_name,
                contents=prompt,
            )
            print("\n🤖 Field Shift Final RAG Output:")
            print("-" * 50)
            print(response.text)
            print("-" * 50)
            success = True
            break
        except Exception as e:
            err_msg = str(e)
            if "503" in err_msg or "UNAVAILABLE" in err_msg or "429" in err_msg:
                wait_time = attempt * 3
                print(f"⏳ Server busy (503). Waiting {wait_time}s before retry...")
                time.sleep(wait_time)
            else:
                print(f"⏭️ Skipping {model_name}: {err_msg[:80]}...")
                break

if not success:
    print("❌ All models failed or servers are too busy. Please try again later.")